# Part 2: Retrieve evidence with microsoft-retriever-1

Cocoarynth's packaging team needs to know when a chocolate bar may carry the organic seal. Before asking a model to write an answer, find the policy passages that support it.

**Mission**
- Connect the existing policies index as an indexed knowledge source.
- Configure `microsoft-retriever-1` through REST and retrieve evidence without an answer-synthesis model.
- Inspect the actual activity and references returned by the service.

## The building blocks

A **knowledge source** connects an existing index to a **knowledge base**. This knowledge base declares `models: [{"kind": "microsoftRetriever1"}]`: there is no customer model endpoint or model key in that declaration.

For this preview, use `retrievalReasoningEffort: {"kind": "minimal"}` and submit the question as a semantic **intent**. With `outputMode: "extractiveData"`, the result is retrieved evidence, not a synthesized answer. Minimal is the request setting; inspect the activity to see what the service actually did.

This first increment uses ordinary search indexes. The graph index for this lab is not available yet, so graph-dependent exercises are deferred. No graph calls or simulated graph results are included.

## Pre-configured environment

Use a feature-enabled preview Search service with the populated `cocoarynth-policies` index and its `semantic-configuration`. The normal self-paced deployment is not assumed to enable this preview or populate a separate preview service. Ask the session owner for a prepared environment.

These samples use raw JSON and Python's built-in HTTP client, not the Azure Search Python SDK. Requests use `2026-11-01-preview` and make billable retrieval calls. They create only notebook-prefixed knowledge sources and bases; they do not provision services, build indexes, or change billing settings.

## Step 1: Set up environment variables

Set `AZURE_SEARCH_PREVIEW_ENDPOINT`, `AZURE_SEARCH_PREVIEW_ADMIN_KEY`, and a unique `AZURE_SEARCH_PREVIEW_RESOURCE_PREFIX` starting with `lab673-`. See the [preview setup](../README.md#preview-rest-notebooks) and the root `.env.sample`. If your prepared indexes have a different prefix, also set `AZURE_SEARCH_PREVIEW_INDEX_PREFIX`.

Open this notebook within the repository and select the existing lab Python kernel. Exported environment variables take precedence over the root `.env` file, so a local test endpoint does not get replaced by the normal deployment configuration. Never paste keys into cells or commit executed notebook outputs.

In [ ]:
import json
import os
from pathlib import Path
import sys

REPO_ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / ".env.sample").is_file() and (directory / "src" / "notebooks").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from within the LAB673 repository.")
sys.path.insert(0, str(REPO_ROOT / "src" / "notebooks"))
from rest_helpers import API_VERSION, SearchRestClient, load_environment, required_env

load_environment(REPO_ROOT)
client = SearchRestClient(
    required_env("AZURE_SEARCH_PREVIEW_ENDPOINT"),
    required_env("AZURE_SEARCH_PREVIEW_ADMIN_KEY"),
    required_env("AZURE_SEARCH_PREVIEW_RESOURCE_PREFIX"),
)
INDEX_PREFIX = os.environ.get("AZURE_SEARCH_PREVIEW_INDEX_PREFIX", "cocoarynth")
POLICIES_INDEX = f"{INDEX_PREFIX}-policies"
SOURCE_NAME = f"{client.resource_prefix}-policy-source"
KNOWLEDGE_BASE_NAME = f"{client.resource_prefix}-extractive"
print(f"REST API: {API_VERSION}; notebook resources: {client.resource_prefix}")

## Step 2: Verify the policies index

`GET /indexes/{name}/docs/$count` confirms that the prepared index contains document chunks. This notebook does not upload documents or generate embeddings.

In [ ]:
count = client.request("GET", f"indexes/{POLICIES_INDEX}/docs/$count")
if not isinstance(count, int) or count <= 0:
    raise RuntimeError("The policies index must be populated before running this exercise.")
print(f"{POLICIES_INDEX}: {count} document chunks")

## Step 3: Create an indexed knowledge source

`PUT /knowledgesources/{name}` connects to the existing index. Search the `chunk` field and return the passage, document title, path, and page numbers so you can inspect the evidence.

The shared helper adds a notebook ownership marker and uses conditional writes. It refuses to overwrite a resource with a different owner. Use your own unique resource prefix, not another attendee's.

In [ ]:
source_definition = {
    "name": SOURCE_NAME,
    "description": "Cocoarynth company policies, including label and marketing claims rules",
    "kind": "searchIndex",
    "searchIndexParameters": {
        "searchIndexName": POLICIES_INDEX,
        "semanticConfigurationName": "semantic-configuration",
        "searchFields": [{"name": "chunk"}],
        "sourceDataFields": [
            {"name": name}
            for name in ("chunk", "blob_path", "title", "page_number_from", "page_number_to")
        ],
    },
}
client.put_owned(f"knowledgesources/{SOURCE_NAME}", source_definition)
print(f"Knowledge source ready: {SOURCE_NAME}")

## Step 4: Create the retriever knowledge base

`PUT /knowledgebases/{name}` selects the built-in retriever and extractive output. This step does not need an Azure OpenAI deployment or key. Any model configuration already used by the prepared index is separate from this knowledge base's model declaration.

In [ ]:
knowledge_base_definition = {
    "name": KNOWLEDGE_BASE_NAME,
    "description": "Retrieve Cocoarynth policy evidence without answer synthesis",
    "knowledgeSources": [{"name": SOURCE_NAME}],
    "models": [{"kind": "microsoftRetriever1"}],
    "retrievalReasoningEffort": {"kind": "minimal"},
    "outputMode": "extractiveData",
}
client.put_owned(f"knowledgebases/{KNOWLEDGE_BASE_NAME}", knowledge_base_definition)
print(f"Knowledge base ready: {KNOWLEDGE_BASE_NAME}")

## Step 5: Retrieve the policy evidence

`POST /knowledgebases('{name}')/retrieve` accepts a semantic intent rather than a chat message for this Minimal request. Ask for references, their source data, and the activity log explicitly.

This is a non-streaming JSON request. `maxRuntimeInSeconds` bounds service work; `maxOutputDocuments` bounds returned evidence. Use `maxOutputSize: 6000` for this preview request.

If the service reports HTTP 402 requiring Standard Knowledge Retrieval, ask the service owner to resolve that prerequisite. Do not treat an error or missing references as a completed exercise.

In [ ]:
QUESTION = (
    "When may a Cocoarynth bar carry the organic seal, and what if the supplier's "
    "organic certificate is pending renewal?"
)
retrieval_request = {
    "intents": [{"type": "semantic", "search": QUESTION}],
    "retrievalReasoningEffort": {"kind": "minimal"},
    "outputMode": "extractiveData",
    "includeActivity": True,
    "knowledgeSourceParams": [{
        "kind": "searchIndex",
        "knowledgeSourceName": SOURCE_NAME,
        "includeReferences": True,
        "includeReferenceSourceData": True,
        "alwaysQuerySource": True,
    }],
    "maxRuntimeInSeconds": 60,
    "maxOutputDocuments": 8,
    "maxOutputSize": 6000,
}
print(json.dumps(retrieval_request, indent=2))
result = client.request("POST", f"knowledgebases('{KNOWLEDGE_BASE_NAME}')/retrieve", retrieval_request)
if not result.get("references"):
    raise RuntimeError("No references returned. Check the prepared index and source configuration.")
print(f"Retrieved {len(result['references'])} references; no answer synthesis was requested.")

### Review the activity log

Inspect the returned `type`, `id`, model metadata, query text, and timing where present. The sequence and number of steps can vary with the question and service response. Activity is evidence of execution, not a guarantee that every relevant passage was found.

In [ ]:
if not result.get("activity"):
    raise RuntimeError("The service did not return the requested activity log.")
print(json.dumps(result["activity"], indent=2))

### Review the references

`sourceData` contains the fields requested on the knowledge source. Use `activitySource` to connect a reference to its retrieval activity when that field is present. Keep the service's reference IDs; do not renumber them.

### Source hunt

Find the policy passage that states the certificate requirement on the packing date. Does it also explain what a pending renewal means for the seal? Which document and page support your conclusion?

In [ ]:
print(json.dumps(result["references"], indent=2))

## Optional: Clean up notebook resources

Leave `CLEAN_UP` false while exploring. Set it to true and run the next cell when finished. It deletes only this notebook's knowledge base and source, in that order. The prepared index and Part 1 resources remain unchanged.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    client.delete_owned(f"knowledgebases/{KNOWLEDGE_BASE_NAME}")
    client.delete_owned(f"knowledgesources/{SOURCE_NAME}")
    print("Removed this notebook's knowledge base and source; the index is unchanged.")
else:
    print("Resources retained. Set CLEAN_UP to True when you finish exploring.")

## Mission complete

You configured a built-in retriever with REST, retrieved policy evidence, and inspected its activity and references without asking for a generated answer.

Continue to [Part 3: Add answer synthesis to the retriever](part3-retriever-answer-synthesis.ipynb) to investigate a multi-document mystery.